# Module 11: Learning the World -- Differentiable Generative Models

---

So far, we've **hand-specified** the generative model: we wrote the A and B matrices
by hand based on our knowledge of the T-maze. But in realistic settings, the agent
must **learn** its own model of the world from experience.

This module covers:

1. **Structure learning vs parameter learning**: knowing the graph vs filling in the numbers
2. **The forward algorithm**: computing $P(o_{1:T} | \text{model})$ via sum-product message passing
3. **Differentiable via JAX**: `jax.grad(NLL)` gives us gradient-based parameter recovery
4. **The EM algorithm connection**: E-step (inference) + M-step (learning)

Because ALF's inference is JAX-native, we can **differentiate through inference**
to learn model parameters. (Note: as of v1.0.0, pymdp is also JAX-first and supports
differentiable learning via pybefit + NumPyro. ALF's approach uses a lightweight
forward-algorithm NLL with `jax.grad` — pedagogically transparent and dependency-free.)

### References

- Smith, Friston & Whyte (2022). Section 4: Learning and adaptation.
- Tschantz, Millidge et al. (2020). Reinforcement Learning through Active Inference.
- Fountas et al. (2020). Deep Active Inference Agents Using Monte-Carlo Methods.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

from alf.generative_model import GenerativeModel
from alf.learning import (
    learn_model, LearningResult, analytic_nll, analytic_nll_single,
    params_to_A, params_to_B, LearnableGenerativeModel,
    generate_data,
)
from alf.agent import AnalyticAgent as ActiveInferenceAgent
from alf.benchmarks.t_maze import (
    build_t_maze_model, TMazeEnv,
    STATE_NAMES, OBS_NAMES, ACTION_NAMES,
    NUM_STATES, NUM_OBS, NUM_ACTIONS,
    ACT_STAY, ACT_CUE, ACT_LEFT, ACT_RIGHT,
)

import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_matrix_heatmap, plot_learning_curve, COLORS,
)

np.set_printoptions(precision=3, suppress=True)
print(f"JAX version: {jax.__version__}")
print("All imports successful.")

## 1. Generate Synthetic Data from a Known HMM

We start by creating a **known** 3-state Hidden Markov Model, generating
observation-action data from it, and then trying to **recover** the parameters.

This is the classic approach: if we can recover known parameters from
synthetic data, we can trust the algorithm on real data.

In [ ]:
# ── Define a 3-state, 4-observation, 2-action HMM ───────────────────────

num_states = 3
num_obs = 4
num_actions = 2

# True A matrix: P(o|s) -- each column sums to 1
A_true = np.array([
    [0.8, 0.1, 0.1],  # obs 0 is likely in state 0
    [0.1, 0.7, 0.1],  # obs 1 is likely in state 1
    [0.05, 0.1, 0.7],  # obs 2 is likely in state 2
    [0.05, 0.1, 0.1],  # obs 3 is an ambiguous signal
])

# True B matrix: P(s'|s, a) for 2 actions
B_true = np.zeros((num_states, num_states, num_actions))

# Action 0: rotate states clockwise (0->1->2->0)
B_true[:, :, 0] = np.array([
    [0.1, 0.1, 0.8],  # s'=0 from s=2
    [0.8, 0.1, 0.1],  # s'=1 from s=0
    [0.1, 0.8, 0.1],  # s'=2 from s=1
])

# Action 1: stay in place (identity-ish)
B_true[:, :, 1] = np.array([
    [0.8, 0.1, 0.1],  # s'=0 mostly from s=0
    [0.1, 0.8, 0.1],  # s'=1 mostly from s=1
    [0.1, 0.1, 0.8],  # s'=2 mostly from s=2
])

# Prior: start in state 0
D_true = np.array([0.8, 0.1, 0.1])

print("True A matrix (likelihood):")
print(A_true)
print(f"\nColumn sums: {A_true.sum(axis=0)}")
print(f"\nTrue B[:,:,0] (rotate):")
print(B_true[:,:,0])
print(f"\nTrue B[:,:,1] (stay):")
print(B_true[:,:,1])

In [ ]:
# ── Generate data from the true model ────────────────────────────────────

# Create a random action sequence
T_data = 200
rng = np.random.RandomState(42)
actions = rng.randint(0, num_actions, size=T_data)

# Generate observations and states
observations, true_states = generate_data(
    A=A_true, B=B_true, D=D_true, actions=actions, seed=42,
)

print(f"Generated {len(observations)} observations and {len(true_states)} states")
print(f"Observation sequence (first 20): {observations[:20]}")
print(f"True state sequence (first 20):  {true_states[:20]}")
print(f"Action sequence (first 20):      {actions[:20]}")

# Verify: observations should correlate with states via A
print(f"\nObservation distribution:")
for o in range(num_obs):
    count = np.sum(observations == o)
    print(f"  obs={o}: {count} ({count/len(observations):.1%})")

## 2. The Analytic NLL is Differentiable

The `analytic_nll` function computes the negative log-likelihood of the
observation sequence using the **forward algorithm** (HMM filtering):

$$-\log P(o_{1:T} | A, B, D) = -\sum_t \log P(o_t | o_{<t}, A, B)$$

Because this is implemented in JAX, we can compute its gradient with
respect to the model parameters. This is the foundation of learning.

In [ ]:
# ── Compute NLL at the true parameters ──────────────────────────────────

# Convert true parameters to log-space (unconstrained)
log_A_true = jnp.array(np.log(np.clip(A_true, 1e-16, None)))
log_B_true = jnp.array(np.log(np.clip(B_true, 1e-16, None)))
D_jnp = jnp.array(D_true)
obs_jnp = jnp.array(observations, dtype=jnp.int32)
# Pad actions to match observations length (last action unused)
act_padded = np.concatenate([actions, [0]])
act_jnp = jnp.array(act_padded, dtype=jnp.int32)

nll_true = analytic_nll_single(log_A_true, log_B_true, D_jnp, obs_jnp, act_jnp)
print(f"NLL at true parameters: {float(nll_true):.4f}")
print(f"Per-step NLL: {float(nll_true) / len(observations):.4f}")

# Compare with random parameters
key = jax.random.PRNGKey(0)
log_A_random = 0.1 * jax.random.normal(key, A_true.shape)
key, subkey = jax.random.split(key)
log_B_random = 0.1 * jax.random.normal(subkey, B_true.shape)

nll_random = analytic_nll_single(log_A_random, log_B_random, D_jnp, obs_jnp, act_jnp)
print(f"\nNLL at random parameters: {float(nll_random):.4f}")
print(f"Per-step NLL: {float(nll_random) / len(observations):.4f}")
print(f"\nTrue parameters are {float(nll_random - nll_true):.1f} nats better.")

In [ ]:
# ── Show that NLL is differentiable ──────────────────────────────────────

grad_fn = jax.grad(analytic_nll_single, argnums=(0, 1))

grad_A, grad_B = grad_fn(log_A_random, log_B_random, D_jnp, obs_jnp, act_jnp)

print("Gradient of NLL w.r.t. log_A_params:")
print(f"  Shape: {grad_A.shape}")
print(f"  Values (first few entries): {grad_A[:2, :].round(3)}")
print(f"  Max abs gradient: {float(jnp.max(jnp.abs(grad_A))):.4f}")

print(f"\nGradient of NLL w.r.t. log_B_params:")
print(f"  Shape: {grad_B.shape}")
print(f"  Max abs gradient: {float(jnp.max(jnp.abs(grad_B))):.4f}")

print("\nThese gradients tell us how to adjust A and B to make the")
print("observations MORE likely -- exactly what gradient descent does.")

## 3. Learning the Model

Now we use `learn_model()` to recover the true A and B matrices from the
observation-action data. This function:

1. Parameterizes A and B in unconstrained log-space
2. Applies column-wise softmax to get valid probability matrices
3. Computes NLL via the forward algorithm
4. Uses `jax.grad` to get gradients
5. Updates parameters via gradient descent (Adam if optax available, else SGD)

In [ ]:
# ── Learn the model ──────────────────────────────────────────────────────

result = learn_model(
    observations=observations,
    actions=act_padded,
    num_obs=num_obs,
    num_states=num_states,
    num_actions=num_actions,
    D=D_true,
    num_epochs=300,
    lr=0.01,
    verbose=True,
)

print(f"\nFinal NLL: {result.loss_history[-1]:.4f}")
print(f"True NLL:  {float(nll_true):.4f}")
print(f"Gap:       {result.loss_history[-1] - float(nll_true):.4f}")

In [ ]:
# ── Plot the learning curve ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(result.loss_history, color=COLORS['aif'], linewidth=2, label='Learning NLL')
ax.axhline(y=float(nll_true), color=COLORS['reward'], linestyle='--',
           linewidth=1.5, label=f'True NLL ({float(nll_true):.1f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('Learning Curve: NLL Decreases as Model Improves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Compare learned vs true A matrices ──────────────────────────────────

A_learned = result.learned_A[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_matrix_heatmap(
    A_true,
    title="True A Matrix",
    row_labels=[f"obs {i}" for i in range(num_obs)],
    col_labels=[f"state {i}" for i in range(num_states)],
    cmap="YlOrRd",
    ax=axes[0],
)

plot_matrix_heatmap(
    A_learned,
    title="Learned A Matrix",
    row_labels=[f"obs {i}" for i in range(num_obs)],
    col_labels=[f"state {i}" for i in range(num_states)],
    cmap="YlOrRd",
    ax=axes[1],
)

plt.suptitle("Likelihood Matrix Recovery: True vs Learned", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Quantitative comparison
mae = np.mean(np.abs(A_true - A_learned))
print(f"Mean Absolute Error (A): {mae:.4f}")
print(f"Max error: {np.max(np.abs(A_true - A_learned)):.4f}")

In [ ]:
# ── Compare learned vs true B matrices ──────────────────────────────────

B_learned = result.learned_B[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for a in range(num_actions):
    plot_matrix_heatmap(
        B_true[:, :, a],
        title=f"True B[:,:,{a}]",
        row_labels=[f"s'={i}" for i in range(num_states)],
        col_labels=[f"s={i}" for i in range(num_states)],
        cmap="Blues",
        ax=axes[0, a],
    )

    plot_matrix_heatmap(
        B_learned[:, :, a],
        title=f"Learned B[:,:,{a}]",
        row_labels=[f"s'={i}" for i in range(num_states)],
        col_labels=[f"s={i}" for i in range(num_states)],
        cmap="Blues",
        ax=axes[1, a],
    )

plt.suptitle("Transition Matrix Recovery: True (top) vs Learned (bottom)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

mae_B = np.mean(np.abs(B_true - B_learned))
print(f"Mean Absolute Error (B): {mae_B:.4f}")

## 4. The Label Switching Problem

A subtle issue with learning HMMs: the learned state labels may be **permuted**
relative to the true labels. State 0 in the learned model might correspond to
state 2 in the true model.

This is called the **label switching problem** and is a fundamental symmetry
of HMMs: any permutation of states with corresponding permutations of A and B
produces the same observation likelihood.

In [ ]:
# ── Demonstrate label switching ──────────────────────────────────────────
#
# Check if the learned model matches after permutation.

from itertools import permutations

best_perm = None
best_error = float('inf')

for perm in permutations(range(num_states)):
    perm = list(perm)
    # Permute A columns and B rows/columns
    A_perm = A_learned[:, perm]
    B_perm = B_learned[np.ix_(perm, perm, range(num_actions))]

    error = np.mean(np.abs(A_true - A_perm)) + np.mean(np.abs(B_true - B_perm))
    if error < best_error:
        best_error = error
        best_perm = perm

print(f"Best permutation: {best_perm}")
print(f"Error before permutation: {np.mean(np.abs(A_true - A_learned)) + np.mean(np.abs(B_true - B_learned)):.4f}")
print(f"Error after permutation:  {best_error:.4f}")

if best_perm != [0, 1, 2]:
    print(f"\nLabel switching detected! The learned states are permuted.")
    print(f"True state 0 -> Learned state {best_perm.index(0)}")
    print(f"True state 1 -> Learned state {best_perm.index(1)}")
    print(f"True state 2 -> Learned state {best_perm.index(2)}")
else:
    print("\nNo label switching -- the learned states match the true states.")
    print("This is lucky! In general, the labels can be arbitrarily permuted.")

## 5. Saddle Points and Local Optima

The NLL landscape for HMM parameters has saddle points and local optima.
Starting from exactly uniform parameters (all zeros in log-space) is a
saddle point where gradients cancel. That's why `learn_model` initializes
with small random perturbations to break symmetry.

Let's see how different initializations affect convergence.

In [ ]:
# ── Multiple random restarts ─────────────────────────────────────────────

n_restarts = 5
curves = {}

for restart in range(n_restarts):
    key = jax.random.PRNGKey(restart * 100)
    key1, key2 = jax.random.split(key)
    init_A = 0.1 * jax.random.normal(key1, (num_obs, num_states))
    init_B = 0.1 * jax.random.normal(key2, (num_states, num_states, num_actions))

    r = learn_model(
        observations=observations,
        actions=act_padded,
        num_obs=num_obs,
        num_states=num_states,
        num_actions=num_actions,
        D=D_true,
        num_epochs=200,
        lr=0.01,
        init_log_A=init_A,
        init_log_B=init_B,
        verbose=False,
    )
    curves[f"Restart {restart}"] = r.loss_history

fig, ax = plt.subplots(figsize=(10, 5))
for name, losses in curves.items():
    ax.plot(losses, linewidth=1.5, alpha=0.8, label=name)

ax.axhline(y=float(nll_true), color='black', linestyle='--',
           linewidth=1, label=f'True NLL')
ax.set_xlabel('Epoch')
ax.set_ylabel('NLL')
ax.set_title('Learning from Multiple Random Initializations')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Different initializations may converge to different local optima.")
print("The best strategy is to run multiple restarts and pick the lowest NLL.")

## 6. Exercise: Learn from Agent Experience

Instead of generating data externally, let's have an ActiveInferenceAgent
interact with the T-maze and then **learn** the world model from its own
experience. This closes the perception-action loop:

Act -> Observe -> Learn -> (use learned model to) Act -> ...

In [ ]:
# ── Collect experience data from AIF agent ──────────────────────────────

gm_tmaze = build_t_maze_model(cue_reliability=0.9, reward_magnitude=3.0, T=2)

all_obs = []
all_actions = []
rng_exp = np.random.RandomState(42)

n_trials = 50
for trial in range(n_trials):
    reward_side = "left" if rng_exp.random() < 0.5 else "right"
    env = TMazeEnv(reward_side=reward_side, cue_reliability=0.9, seed=42 + trial)
    agent = ActiveInferenceAgent(gm_tmaze, gamma=4.0, seed=42 + trial)
    agent.reset()

    obs = env.reset()
    trial_obs = [obs]
    trial_actions = []

    for step in range(2):
        action, _ = agent.step([obs])
        obs, reward, done = env.step(action)
        trial_actions.append(action)
        trial_obs.append(obs)
        if done:
            break

    all_obs.extend(trial_obs)
    all_actions.extend(trial_actions)

print(f"Collected {len(all_obs)} observations and {len(all_actions)} actions")
print(f"from {n_trials} T-maze trials.")

In [ ]:
# ── Learn the T-maze model from agent experience ────────────────────────

# Pad actions to match observations length
obs_exp = np.array(all_obs)
act_exp = np.array(all_actions)

# Align: observations has one more entry than actions
# Use all observations, pad actions
min_len = min(len(obs_exp), len(act_exp) + 1)
obs_exp = obs_exp[:min_len]
act_exp_padded = np.concatenate([act_exp[:min_len-1], [0]])

result_tmaze = learn_model(
    observations=obs_exp,
    actions=act_exp_padded,
    num_obs=NUM_OBS,
    num_states=NUM_STATES,  # Using true number of states
    num_actions=NUM_ACTIONS,
    D=gm_tmaze.D[0],
    num_epochs=300,
    lr=0.01,
    verbose=True,
)

print(f"\nFinal NLL: {result_tmaze.loss_history[-1]:.4f}")

In [ ]:
# ── Compare learned A with true T-maze A ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plot_matrix_heatmap(
    gm_tmaze.A[0],
    title="True T-Maze A Matrix",
    row_labels=OBS_NAMES,
    col_labels=STATE_NAMES,
    cmap="YlOrRd",
    ax=axes[0],
)

plot_matrix_heatmap(
    result_tmaze.learned_A[0],
    title="Learned A Matrix (from agent experience)",
    row_labels=OBS_NAMES,
    col_labels=[f"s{i}" for i in range(NUM_STATES)],
    cmap="YlOrRd",
    ax=axes[1],
)

plt.tight_layout()
plt.show()

print("Note: the learned A may have permuted state labels (label switching).")
print("The structure should be similar even if the columns are reordered.")

## 7. Exercise: Too Many or Too Few States?

What happens when we assume the **wrong number** of hidden states?

- **Too few states**: the model can't capture the true dynamics (underfitting)
- **Too many states**: some states are redundant (overfitting, degenerate solutions)

In [ ]:
# ── Experiment: varying the assumed number of states ─────────────────────

state_counts = [2, 3, 4, 5, 6]
final_nlls = []
all_loss_curves = {}

for ns in state_counts:
    D_ns = np.ones(ns) / ns
    r = learn_model(
        observations=observations,
        actions=act_padded,
        num_obs=num_obs,
        num_states=ns,
        num_actions=num_actions,
        D=D_ns,
        num_epochs=300,
        lr=0.01,
        verbose=False,
    )
    final_nlls.append(r.loss_history[-1])
    all_loss_curves[f"{ns} states"] = r.loss_history

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Learning curves
for name, losses in all_loss_curves.items():
    axes[0].plot(losses, linewidth=1.5, label=name)
axes[0].axhline(y=float(nll_true), color='black', linestyle='--', linewidth=1, label='True NLL')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NLL')
axes[0].set_title('Learning Curves: Different State Counts')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Final NLL vs number of states
axes[1].bar(state_counts, final_nlls, color=COLORS['aif'], alpha=0.8)
axes[1].axhline(y=float(nll_true), color=COLORS['reward'], linestyle='--',
                linewidth=1.5, label='True NLL (3 states)')
axes[1].set_xlabel('Assumed Number of States')
axes[1].set_ylabel('Final NLL')
axes[1].set_title('Model Selection: Which State Count Fits Best?')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nBest NLL by state count:")
for ns, nll in zip(state_counts, final_nlls):
    marker = " <-- true" if ns == num_states else ""
    print(f"  {ns} states: NLL = {nll:.4f}{marker}")

print(f"\nWith too few states, the model can't capture the dynamics.")
print(f"With too many states, NLL may improve slightly (overfitting) but")
print(f"some states become redundant duplicates.")

## Summary

In this notebook you have:

1. **Generated synthetic data** from a known 3-state HMM
2. **Computed the differentiable NLL** and verified gradients exist
3. **Learned A and B matrices** via gradient descent on the forward-algorithm NLL
4. **Visualized** the recovery of true parameters (with label-switching caveats)
5. **Explored** saddle points, local optima, and the benefit of random restarts
6. **Learned from agent experience**: closing the perception-action-learning loop
7. **Investigated model selection**: too few vs too many hidden states

### Key Takeaway

Because ALF implements inference in JAX, the entire forward algorithm is
**automatically differentiable**. This means we can learn generative model
parameters by gradient descent. As of v1.0.0, pymdp also supports JAX-native
differentiable learning (via pybefit + NumPyro). ALF's approach here is a
lightweight alternative using pure `jax.grad` through the forward algorithm.

### Next

In Module 12, we scale to **high-dimensional observations** using neural
network encoders -- Deep Active Inference.